In [ ]:
import marimo as mo

MINi project ML
# Explainable AI  for Loan Approval Decisions

This is my mini project. The idea is simple :- Banks use ML to decide if a loan should be
approved or not, but models like random forest dont tell us WHY they said no.
So here I train a few models on the loan prediction dataset and then use SHAP and LIME
to explain the decisions.

dataset = train.csv (614 rows, from the analytics vidhya loan prediction problem)
test.csv is also there, I use it at the end to predict for new customers.

## Problem statement and objectives

**problem** - banks get thousands of loan applications and now ML models decide approve/reject.
models like random forest are accurate but they are a black box - if a customer gets rejected
nobody can say "WHY ?". in banking that is a problem because the customer has a right to know,
and the bank also has to be able to justify the decision.

**objectives**
1. do the EDA and preprocessing on the loan prediction dataset (missing values, encoding)
2. train and compare 4 models - logistic regression, decision tree, SVM, random forest
3. evaluate with accuracy, precision, recall and F1 (not only accuracy, because the data is imbalanced)
4. use SHAP and LIME to explain the model globally (which features matter) and locally (why this one customer)
5. build a small working prototype that takes a new customer's details and prints the decision + top reasons
6. predict the unlabeled test.csv customers as a final demo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data = pd.read_csv("train.csv")

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

#import shap
#import lime
#import lime.lime_tabular

In [ ]:
data.info()

Check Shape and Missing Values

In [ ]:
data.apply(lambda x: sum(x.isnull()),axis=0) # checking missing values in each column of train dataset)

In [ ]:
data['Gender'].value_counts()

# Visualize how different features affect the 'Loan_Status'

In [ ]:
plt.figure(figsize=(12, 8))
plt.subplot(2, 2, 1)
sns.countplot(x='Loan_Status',data=data,palette='Set2')
plt.title('Loan Approval Status Count')
 #2. Credit History vs Loan Status
plt.subplot(2, 2, 2)
sns.countplot(x='Credit_History', hue='Loan_Status',data=data,palette='Set1')
plt.title('Credit History vs Loan Status')
#3. Education vs Loan Status
plt.subplot(2, 2, 3)
sns.countplot(x='Education', hue='Loan_Status',data=data,palette='Pastel1')
plt.title('Education vs Loan Status')
#4. Loan Amount Distribution
plt.subplot(2, 2, 4)
sns.histplot(data['LoanAmount'].dropna(),kde=True,color='teal')
plt.title('Loan Amount Distribution')
plt.tight_layout()
plt.show()

so from the plots - most people got the loan (Y is more than N), credit history = 1 people
get approved almost always, graduates are a bit better than non graduates, and loan amount
is skewed to the right (few people asking very big loans).

##  Data Cleaning and Handling  Missing Values from the dataset

In [ ]:
data.Gender = data.Gender.fillna('Male')
data['Married'].value_counts()

In [ ]:
data.Married = data.Married.fillna('Yes')
data['Dependents'].value_counts()

In [ ]:
data.Dependents = data.Dependents.fillna('0')
data['Self_Employed'].value_counts()

In [ ]:
data.Self_Employed=data.Self_Employed.fillna('No')
data.LoanAmount=data.LoanAmount.fillna(data.LoanAmount.mean())
data['Loan_Amount_Term'].value_counts()

In [ ]:
data.Loan_Amount_Term=data.Loan_Amount_Term.fillna(360.0)
data['Credit_History'].value_counts()

In [ ]:
data.Credit_History=data.Credit_History.fillna(1.0)
data.apply(lambda x: sum(x.isnull()),axis=0)

In [ ]:
data.head()

In [ ]:
data_1 = data.drop('Loan_ID', axis=1)
X = data_1.iloc[:, 0:11]
y = data_1.iloc[:, 11]
X.head()

In [ ]:
y

In [ ]:
# Drop ID columns if present
if 'Loan_ID' in data_1.columns:
    data_2 = data_1.drop('Loan_ID', axis=1)
if 'Unnamed: 0' in data_2.columns:
    data_2 = data_2.drop('Unnamed: 0', axis=1)
# Fill categorical missing values with Mode
data_2['Gender'] = data_2['Gender'].fillna(data_2['Gender'].mode()[0])
data_2['Married'] = data_2['Married'].fillna(data_2['Married'].mode()[0])
data_2['Dependents'] = data_2['Dependents'].fillna(data_2['Dependents'].mode()[0])
data_2['Self_Employed'] = data_2['Self_Employed'].fillna(data_2['Self_Employed'].mode()[0])
data_2['Credit_History'] = data_2['Credit_History'].fillna(data_2['Credit_History'].mode()[0])
# Fill numerical missing values with Median
data_2['LoanAmount'] = data_2['LoanAmount'].fillna(data_2['LoanAmount'].median())
data_2['Loan_Amount_Term'] = data_2['Loan_Amount_Term'].fillna(data_2['Loan_Amount_Term'].median())
# Verify that no missing values remain
print('Remaining Missing Values:\n', data_2.isnull().sum())

### encoding the text columns

models only understand numbers so Gender, Married etc have to be converted.
Dependents has '3+' in it which is text, so I make it 3.

In [ ]:
data_2['Dependents'] = data_2['Dependents'].replace('3+', 3).astype(int)
data_2['Gender'] = data_2['Gender'].map({'Male': 1, 'Female': 0})
data_2['Married'] = data_2['Married'].map({'Yes': 1, 'No': 0})
data_2['Education'] = data_2['Education'].map({'Graduate': 1, 'Not Graduate': 0})
data_2['Self_Employed'] = data_2['Self_Employed'].map({'Yes': 1, 'No': 0})
data_2['Property_Area'] = data_2['Property_Area'].map({'Rural': 0, 'Semiurban': 1, 'Urban': 2})
data_2['Loan_Status'] = data_2['Loan_Status'].map({'Y': 1, 'N': 0})
data_2.head()  # Y = approved

In [ ]:
# which features are related to each other
plt.figure(figsize=(10, 7))
sns.heatmap(data_2.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

### Train test split

80% for training and 20% for testing. scaling is done only for logistic regression and SVM
because they use distances, trees dont need it. I keep the unscaled data for SHAP/LIME so
the explanations show real income and loan amount values.

In [ ]:
X_1 = data_2.drop('Loan_Status', axis=1)
y_1 = data_2['Loan_Status']
X_train, X_test, y_train, y_test = train_test_split(X_1, y_1, test_size=0.2, random_state=42, stratify=y_1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Training set size:', X_train.shape)
print('Testing set size:', X_test.shape)

## Trying the models

4 models - logistic regression, decision tree, SVM and random forest.

In [ ]:
# 1. logistic regression
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)

# 2. decision tree
dt_model = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

# 3. SVM
svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train)
svm_pred = svm_model.predict(X_test_scaled)

# 4. random forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("all 4 models trained")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report


rows = []
for name, pred in [('Logistic Regression', lr_pred), ('Decision Tree', dt_pred),
                   ('SVM', svm_pred), ('Random Forest', rf_pred)]:
    rows.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, pred), 4),
        'Precision': round(precision_score(y_test, pred), 4),
        'Recall': round(recall_score(y_test, pred), 4),
        'F1-Score': round(f1_score(y_test, pred), 4)
    })

results = pd.DataFrame(rows)
results.to_csv('results.csv', index=False)   # saving so I can put it in the report
results

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(x='Model', y='Accuracy', data=results, palette='viridis')
plt.ylim(0.6, 1.0)
plt.title('Accuracy of the 4 models')
plt.ylabel('Accuracy')
plt.show()

In [ ]:
cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Rejected','Approved'], yticklabels=['Rejected','Approved'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Random Forest confusion matrix')
plt.show()

print(classification_report(y_test, rf_pred, target_names=['Rejected','Approved']))

## Explainable AI part

random forest is 100 trees so we cant read it. Three things here -
1. decision tree plot (this one we CAN read)
2. SHAP - which features matter overall + one customer
3. LIME - one customer again, to check SHAP is not lying

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(dt_model, feature_names=X_1.columns, class_names=['Rejected', 'Approved'], filled=True, rounded=True, fontsize=10)
plt.title('Decision tree rules')
plt.show()

### SHAP

In [ ]:
import shap

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer(X_test)

# shap gives values for both classes, we want class 1 (approved)
if len(shap_values.shape) == 3:
    shap_values_approved = shap_values[:, :, 1]
else:
    shap_values_approved = shap_values

shap.summary_plot(shap_values_approved, X_test, show=False)
plt.title('SHAP summary - what pushes the loan decision')
plt.show()

how to read this - top feature is the most important one. red dot = high value of that
feature, blue = low. if the red dots are on the right side it means high value pushes
towards approval. credit history is clearly the biggest thing here, way bigger than
income or loan amount.

In [ ]:
# now explain ONE customer from the test set
i = 0
print("customer number", i)
print(X_test.iloc[i])
print("prediction:", rf_pred[i], "(1 = approved, 0 = rejected)")

shap.plots.waterfall(shap_values_approved[i])

### LIME

In [ ]:
import lime
import lime.lime_tabular
lime_explainer = lime.lime_tabular.LimeTabularExplainer(training_data=np.array(X_train), feature_names=list(X_1.columns), class_names=['Rejected', 'Approved'], mode='classification', random_state=42)
lime_exp = lime_explainer.explain_instance(data_row=X_test.iloc[i].values, predict_fn=rf_model.predict_proba)
fig = lime_exp.as_pyplot_figure()
plt.title('LIME explanation for customer ' + str(i))
plt.tight_layout()
plt.show()

In [ ]:
lime_exp.as_list()

## prototype - check any new customer

small function where I can type a new customer and it tells approved/rejected +
the main reasons from SHAP.

In [ ]:
def check_loan_application(customer):
    new_df = pd.DataFrame([customer])[X_1.columns]
    pred = rf_model.predict(new_df)[0]
    prob = rf_model.predict_proba(new_df)[0]
    if pred == 1:
        print('Loan Decision : APPROVED')
        print('Confidence    :', round(prob[1] * 100, 2), '%')
    else:
        print('Loan Decision : REJECTED')
        print('Confidence    :', round(prob[0] * 100, 2), '%')
    sv = explainer(new_df)
    if len(sv.shape) == 3:
        vals = sv[0, :, 1].values
    elif len(sv.shape) == 2:  # shap for this new customer
        vals = sv.values[0]
    else:
        vals = sv.values
    reasons = pd.DataFrame({'Feature': X_1.columns, 'Value': new_df.iloc[0].values, 'Impact': vals})
    reasons['abs'] = reasons['Impact'].abs()
    reasons = reasons.sort_values('abs', ascending=False)
    print('main reasons:')
    for _, r in reasons.head(4).iterrows():
        if r['Impact'] > 0:
            print('  ', r['Feature'], '=', r['Value'], '-> towards APPROVAL', round(r['Impact'], 3))
        else:
            print('  ', r['Feature'], '=', r['Value'], '-> towards REJECTION', round(r['Impact'], 3))
    print()
c1 = {'Gender': 1, 'Married': 1, 'Dependents': 0, 'Education': 1, 'Self_Employed': 0, 'ApplicantIncome': 5500, 'CoapplicantIncome': 1500, 'LoanAmount': 130, 'Loan_Amount_Term': 360, 'Credit_History': 1.0, 'Property_Area': 1}
print('Test case 1')
check_loan_application(c1)
c2 = {'Gender': 1, 'Married': 0, 'Dependents': 2, 'Education': 0, 'Self_Employed': 1, 'ApplicantIncome': 2500, 'CoapplicantIncome': 0, 'LoanAmount': 250, 'Loan_Amount_Term': 180, 'Credit_History': 0.0, 'Property_Area': 0}
print('Test case 2')
# customer 1 - good credit history, decent income
# customer 2 - no credit history and asking a big loan
check_loan_application(c2)

## Predicting for the test.csv customers

test.csv has no Loan_Status column (thats the point), so I clean it the same way and
save the predictions to loan_predictions.csv.

In [ ]:
test = pd.read_csv("test.csv")
print(test.shape)
test.head()

In [ ]:
# same cleaning as train
ids = test['Loan_ID']
test_1 = test.drop('Loan_ID', axis=1)
for col in ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Credit_History']:
    test_1[col] = test_1[col].fillna(test_1[col].mode()[0])
test_1['LoanAmount'] = test_1['LoanAmount'].fillna(test_1['LoanAmount'].median())
test_1['Loan_Amount_Term'] = test_1['Loan_Amount_Term'].fillna(test_1['Loan_Amount_Term'].median())
test_1['Dependents'] = test_1['Dependents'].replace('3+', 3).astype(int)
test_1['Gender'] = test_1['Gender'].map({'Male': 1, 'Female': 0})
test_1['Married'] = test_1['Married'].map({'Yes': 1, 'No': 0})
test_1['Education'] = test_1['Education'].map({'Graduate': 1, 'Not Graduate': 0})
test_1['Self_Employed'] = test_1['Self_Employed'].map({'Yes': 1, 'No': 0})
test_1['Property_Area'] = test_1['Property_Area'].map({'Rural': 0, 'Semiurban': 1, 'Urban': 2})
print('missing left:', test_1.isnull().sum().sum())

In [ ]:
test_pred = rf_model.predict(test_1)
out = pd.DataFrame({'Loan_ID': ids, 'Loan_Status': test_pred})
out['Loan_Status'] = out['Loan_Status'].map({1: 'Y', 0: 'N'})
out.to_csv('loan_predictions.csv', index=False)
print(out['Loan_Status'].value_counts())
out.head(10)

## Results

ok so after running everything -

**models (on the 123 test customers):**
- logistic regression - 86.18% accuracy, F1 0.908  <- best one
- SVM - 85.37%, F1 0.903
- decision tree - 84.55%, F1 0.898
- random forest - 84.55%, F1 0.898

all four are basically the same (within 1.6%), random forest is NOT better here. I still
explain the random forest because it is the black box one, that was the point of the project.

one thing I noticed - recall for "approved" is 0.99 for every model but recall for
"rejected" is only 0.53. the data is 69% approved so the models are scared to reject people.

**SHAP** said credit history is the biggest feature by far (average |shap| = 0.15, the next
one loan amount is only 0.027, so like 5-6 times bigger). then loan amount, applicant income,
property area. **LIME** said the same thing for the same customer, so both methods agree.

**prototype:**
- customer with credit history 1 + income 5500 + loan 130 -> APPROVED (83.03%), main reason credit history
- customer with credit history 0 + loan 250 -> REJECTED (68.68%), main reason credit history (-0.361)

**test.csv** -> 367 customers, model said 306 Y and 61 N, saved in loan_predictions.csv

files made by this notebook: results.csv (model comparison table), loan_predictions.csv